In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import json
import os
from datetime import datetime

In [14]:
X_FEATURES_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\processed\\finalX_features.csv"
INPUT_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready"
CLEANED_DATASET_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\cleaned_dataset.csv"

OUTPUT_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data "
METADATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\metadata"
DOCS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs"

RANDOM_SEED = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.1

# Create directories if missing
for path in [OUTPUT_PATH, METADATA_PATH, DOCS_PATH]:
    os.makedirs(path, exist_ok=True)

In [15]:
print("=" * 60)
print("EcoPackAI - Train/Test Split & Cross-Validation")
print("=" * 60)

print("\n[1/6] Loading ML-ready datasets...")

try:
    X = pd.read_csv(X_FEATURES_PATH)
    y_cost = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_co2.csv")
    y_co2 = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_cost.csv")
    y_sustainability = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_sustainability.csv")
    
    print(f"✓ Features loaded: {X.shape}")
    print(f"✓ Target - Cost: {y_cost.shape}")
    print(f"✓ Target - CO2: {y_co2.shape}")
    print(f"✓ Target - Sustainability: {y_sustainability.shape}")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("\nPlease run data cleaning script first")
    exit(1)

EcoPackAI - Train/Test Split & Cross-Validation

[1/6] Loading ML-ready datasets...
✓ Features loaded: (606000, 23)
✓ Target - Cost: (606000, 1)
✓ Target - CO2: (606000, 1)
✓ Target - Sustainability: (606000, 1)


In [16]:
print("\n[2/6] Validating data integrity...")

# Check for missing values
missing_X = X.isnull().sum().sum()
missing_y_cost = y_cost.isnull().sum().sum()
missing_y_co2 = y_co2.isnull().sum().sum()
missing_y_sust = y_sustainability.isnull().sum().sum()

if missing_X > 0 or missing_y_cost > 0 or missing_y_co2 > 0 or missing_y_sust > 0:
    print(f"⚠ Warning: Found missing values!")
    print(f"  Features: {missing_X}")
    print(f"  Cost target: {missing_y_cost}")
    print(f"  CO2 target: {missing_y_co2}")
    print(f"  Sustainability target: {missing_y_sust}")
else:
    print("✓ No missing values found")

# Check shape consistency
if len(X) == len(y_cost) == len(y_co2) == len(y_sustainability):
    print(f"✓ All datasets have consistent length: {len(X)} rows")
else:
    print("❌ Error: Dataset lengths don't match!")
    exit(1)


[2/6] Validating data integrity...
✓ No missing values found
✓ All datasets have consistent length: 606000 rows


In [17]:
# First split: test set
X_temp, X_test, y_cost_temp, y_cost_test, y_co2_temp, y_co2_test, y_sust_temp, y_sust_test = train_test_split(
    X, y_cost, y_co2, y_sustainability,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True
)

# Second split: validation set from training
val_size_adjusted = VALIDATION_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_cost_train, y_cost_val, y_co2_train, y_co2_val, y_sust_train, y_sust_val = train_test_split(
    X_temp, y_cost_temp, y_co2_temp, y_sust_temp,
    test_size=val_size_adjusted,
    random_state=RANDOM_SEED,
    shuffle=True
)

print(f"\nSplit completed:")
print(f"  Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")


Split completed:
  Training set: 424200 samples (70.0%)
  Validation set: 60600 samples (10.0%)
  Test set: 121200 samples (20.0%)


In [18]:
df_original = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\cleaned_dataset.csv")

train_indices = X_train.index
val_indices = X_val.index
test_indices = X_test.index

if 'product_category' in df_original.columns:
    print("\nProduct Category Distribution:")
    train_dist = df_original.loc[train_indices, 'product_category'].value_counts(normalize=True)
    test_dist = df_original.loc[test_indices, 'product_category'].value_counts(normalize=True)
    
    print("  Train set:")
    for cat, pct in train_dist.items():
        print(f"    {cat}: {pct*100:.1f}%")
    
    print("  Test set:")
    for cat, pct in test_dist.items():
        print(f"    {cat}: {pct*100:.1f}%")


Product Category Distribution:
  Train set:
    Food: 22.9%
    Electronics: 19.7%
    Cosmetics: 18.1%
    Paper Product: 15.4%
    Pharmacy: 13.8%
    Drinkware: 9.9%
     Cosmetics: 0.3%
  Test set:
    Food: 22.7%
    Electronics: 19.4%
    Cosmetics: 18.3%
    Paper Product: 15.5%
    Pharmacy: 14.0%
    Drinkware: 9.8%
     Cosmetics: 0.3%


In [19]:
print("\nTarget Variable Statistics:")
print(f"\nCost (USD):")
print(f"  Train - Mean: ${y_cost_train.iloc[:,0].mean():.2f}, Std: ${y_cost_train.iloc[:,0].std():.2f}")
print(f"  Val   - Mean: ${y_cost_val.iloc[:,0].mean():.2f}, Std: ${y_cost_val.iloc[:,0].std():.2f}")
print(f"  Test  - Mean: ${y_cost_test.iloc[:,0].mean():.2f}, Std: ${y_cost_test.iloc[:,0].std():.2f}")

print(f"\nCO2 Footprint (kg):")
print(f"  Train - Mean: {y_co2_train.iloc[:,0].mean():.3f}, Std: {y_co2_train.iloc[:,0].std():.3f}")
print(f"  Val   - Mean: {y_co2_val.iloc[:,0].mean():.3f}, Std: {y_co2_val.iloc[:,0].std():.3f}")
print(f"  Test  - Mean: {y_co2_test.iloc[:,0].mean():.3f}, Std: {y_co2_test.iloc[:,0].std():.3f}")

print("\nSustainability Score:")
print(f"  Train - Mean: {y_sust_train.iloc[:,0].mean():.2f}, Std: {y_sust_train.iloc[:,0].std():.2f}")
print(f"  Val   - Mean: {y_sust_val.iloc[:,0].mean():.2f}, Std: {y_sust_val.iloc[:,0].std():.2f}")
print(f"  Test  - Mean: {y_sust_test.iloc[:,0].mean():.2f}, Std: {y_sust_test.iloc[:,0].std():.2f}")


Target Variable Statistics:

Cost (USD):
  Train - Mean: $1.19, Std: $0.75
  Val   - Mean: $1.18, Std: $0.74
  Test  - Mean: $1.19, Std: $0.75

CO2 Footprint (kg):
  Train - Mean: 8.926, Std: 7.083
  Val   - Mean: 8.921, Std: 7.059
  Test  - Mean: 8.915, Std: 7.079

Sustainability Score:
  Train - Mean: 65.41, Std: 9.98
  Val   - Mean: 65.44, Std: 9.92
  Test  - Mean: 65.43, Std: 9.97


In [20]:
X_train.to_csv(f"{OUTPUT_PATH}X_train.csv", index=False)
y_cost_train.to_csv(f"{OUTPUT_PATH}y_cost_train.csv", index=False)
y_co2_train.to_csv(f"{OUTPUT_PATH}y_co2_train.csv", index=False)
y_sust_train.to_csv(f"{OUTPUT_PATH}y_sustainability_train.csv", index=False)

# Validation sets
X_val.to_csv(f"{OUTPUT_PATH}X_val.csv", index=False)
y_cost_val.to_csv(f"{OUTPUT_PATH}y_cost_val.csv", index=False)
y_co2_val.to_csv(f"{OUTPUT_PATH}y_co2_val.csv", index=False)
y_sust_val.to_csv(f"{OUTPUT_PATH}y_sustainability_val.csv", index=False)

# Test sets
X_test.to_csv(f"{OUTPUT_PATH}X_test.csv", index=False)
y_cost_test.to_csv(f"{OUTPUT_PATH}y_cost_test.csv", index=False)
y_co2_test.to_csv(f"{OUTPUT_PATH}y_co2_test.csv", index=False)
y_sust_test.to_csv(f"{OUTPUT_PATH}y_sustainability_test.csv", index=False)

print("✓ All split datasets saved")



✓ All split datasets saved


In [21]:

split_metadata = {
    "split_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "train_size": 1 - TEST_SIZE - VALIDATION_SIZE,
    "total_samples": len(X),
    "train_samples": len(X_train),
    "validation_samples": len(X_val),
    "test_samples": len(X_test),
    "feature_count": X.shape[1],
    "feature_names": X.columns.tolist(),
    "target_variables": {
        "cost_prediction": {
            "column": "cost_per_unit_usd",
            "train_mean": float(y_cost_train.iloc[:,0].mean()),
            "train_std": float(y_cost_train.iloc[:,0].std()),
            "test_mean": float(y_cost_test.iloc[:,0].mean()),
            "test_std": float(y_cost_test.iloc[:,0].std())
        },
        "co2_prediction": {
            "column": "carbon_footprint",
            "train_mean": float(y_co2_train.iloc[:,0].mean()),
            "train_std": float(y_co2_train.iloc[:,0].std()),
            "test_mean": float(y_co2_test.iloc[:,0].mean()),
            "test_std": float(y_co2_test.iloc[:,0].std())
        },
        "sustainability_score": {
            "column": "overall_sustainability_score",
            "train_mean": float(y_sust_train.iloc[:,0].mean()),
            "train_std": float(y_sust_train.iloc[:,0].std()),
            "test_mean": float(y_sust_test.iloc[:,0].mean()),
            "test_std": float(y_sust_test.iloc[:,0].std())
        }
    },
    "cross_validation_strategy": {
        "method": "5-Fold Cross-Validation",
        "n_folds": 5,
        "shuffle": True,
        "stratified": False
    }
}

# Save metadata
with open(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\metadata\\split_metadata.json", 'w') as f:
    json.dump(split_metadata, f, indent=2)

print(f"✓ Metadata saved: {METADATA_PATH}split_metadata.json")

print("\n✅ Train/Test split complete! Ready for model training.")

✓ Metadata saved: C:\Users\sneha\Desktop\ecopackai\Data\ml\metadatasplit_metadata.json

✅ Train/Test split complete! Ready for model training.
